# VC5 · Demo en directo — Smart City
## "Alcalde, ¿dónde ponemos el presupuesto? ...déjame ver."

**Esto NO es la actividad evaluable.** Es la demostración de la videoconferencia del NF5.
Tu actividad hace un informe estratégico de **un juego online**; aquí, de una ciudad. Mismo
trabajo, otros datos. **No se entrega.**

**Antes de ejecutar nada**, una sola vez, desde la raíz del repositorio:

```bash
python demo/nf5_smartcity/preparar_demo.py
```

---

### El encargo

Llevamos cuatro núcleos de ingeniería impecable. Guardamos bien (NF1), procesamos a escala
(NF2), sabemos que el dato es fiable (NF3) y lo vigilamos en el tiempo (NF4). Y esta mañana el
alcalde entra en la oficina y pregunta lo único que le importa:

> *"Tengo que decidir dónde va el presupuesto de calidad del aire del año que viene. Tenéis dos
> años de datos de toda la ciudad. **¿Dónde lo pongo?**"*

Toda nuestra fontanería no vale nada si no sabemos responder a eso. Vamos a intentarlo — y a
tropezar en cada paso con una lección del núcleo.

In [ ]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

try:
    BASE = Path(__file__).parent
except NameError:
    BASE = Path.cwd()
    if BASE.name != "nf5_smartcity":
        BASE = BASE / "demo" / "nf5_smartcity"
RAW = BASE / "raw"

assert (RAW / "eventos.parquet").exists(), (
    "No encuentro los datos. Ejecuta una vez, desde la raiz del repo:\n"
    "    python demo/nf5_smartcity/preparar_demo.py"
)
print("Datos listos en:", BASE)

---
# ACTO 1 · El error de novato: conectar la herramienta al data lake

La tentacion es abrir Power BI (o Tableau), apuntarlo a los **2 millones de eventos** en bruto
y empezar a hacer clic. Vamos a simular lo que pasa cuando el alcalde, a tu lado, filtra por
una zona: la herramienta lanza una consulta que **escanea el data lake entero**.

In [ ]:
eventos = str(RAW / "eventos.parquet")

def cron(fn):
    t = time.perf_counter(); r = fn(); return time.perf_counter() - t, r

# El alcalde hace clic en "zona Centro". La herramienta escanea los 2M para responder.
t, _ = cron(lambda: duckdb.sql(f"SELECT count(*) FROM '{eventos}' WHERE id_zona = 1").fetchone())
print(f"1 clic (filtrar por zona) sobre 2M de filas: {t*1000:.0f} ms")

# Pero un dashboard no hace 1 consulta. Hace una por panel, por filtro, por cada clic.
t_dash, _ = cron(lambda: [duckdb.sql(
    f"SELECT tipo, count(*) FROM '{eventos}' WHERE id_zona={z} GROUP BY tipo").fetchall()
    for z in range(1, 6)])
print(f"Un dashboard con ~5 paneles interactivos:     {t_dash*1000:.0f} ms por refresco")

DuckDB es rapidísimo, así que aquí son milisegundos. Pero piensa en la realidad de producción:
esto no es DuckDB local, es la herramienta de BI contra un data lake **en la nube**, con
**cientos de millones** de filas, y **cada clic del alcalde** vuelve a escanearlo.

> **El problema no es técnico, es humano.** §5.3 lo dice sin rodeos: *un analista que espera 3
> segundos explora; uno que espera 3 minutos, no*. Deja de hacerse preguntas. Y un dashboard que
> no invita a preguntar **ha fracasado**, por correctos que sean sus datos. La lentitud no
> molesta: **mata el análisis**. Si el alcalde espera, deja de preguntar y decide por intuición
> — que es justo lo que veniamos a evitar.

Y hay un coste literal: si el motor cobra por bytes escaneados (NF1 §1.4), **cada clic es
dinero**. La solución tiene nombre y es la pieza central del núcleo: un **Data Mart**.

---
# ACTO 2 · El Data Mart: de 2 millones de filas a 5

Un Data Mart es una porción **pequeña, agregada y optimizada** para responder a **un dominio de
preguntas**. El del alcalde es: *"qué zona necesita el presupuesto?"*. Así que agregamos **por
zona**, una sola vez, con toda la potencia de un motor analítico.

In [ ]:
alertas = str(RAW / "alertas.parquet")
zonas = pd.read_csv(RAW / "zonas.csv")

# ETL: Extraer (parquet), Transformar (agregar por zona), Cargar (un CSV diminuto).
mart = duckdb.sql(f'''
    SELECT
        e.id_zona,
        count(*)                                             AS n_eventos,
        (SELECT count(*) FROM '{alertas}' a
                 WHERE a.id_zona = e.id_zona)                AS n_alertas,
        (SELECT count(*) FROM '{alertas}' a
                 WHERE a.id_zona = e.id_zona
                   AND a.severidad = 'critica')              AS n_criticas
    FROM '{eventos}' e
    GROUP BY e.id_zona
    ORDER BY e.id_zona
''').df()

mart = mart.merge(zonas[["id_zona", "nombre"]], on="id_zona")
print("El data mart entero:\n")
print(mart.to_string(index=False))
print(f"\n2.000.000 de eventos  ->  {len(mart)} filas. ESO es lo que consume el dashboard.")

Cinco filas. Eso es lo que la herramienta de BI carga en memoria y sirve al instante. **Por eso
vuela.** Y no es un montón de números sueltos: es un **esquema en estrella** de libro.

- **`fact` (hechos): lo que mides.** `n_eventos`, `n_alertas`, `n_criticas` — numeros que
  **sumas**. La tabla grande.
- **`dim` (dimension): el contexto.** `nombre`, `poblacion` de la zona — por lo que **agrupas**.
  La tabla pequena.
- **El grano:** *una fila por zona*. Es la decisión que no se deshace: este mart puede responder
  *"qué zona?"* y **no** *"a qué hora?"* — esa columna la agregamos y ya no está.

> **La regla para clasificar:** si lo **sumarías**, es un hecho; si lo usarías para **agrupar o
> filtrar**, es una dimensión. `n_alertas` -> hecho. `nombre de zona` -> dimensión.

Guardamos el mart. **Este CSV es lo que cargarías en Power BI o en Tableau** — no los 2 millones
de filas.

In [ ]:
MART = BASE / "data_mart"
MART.mkdir(exist_ok=True)
mart.to_csv(MART / "fact_zona.csv", index=False)
print("Guardado:", (MART / "fact_zona.csv"))
print(f"Tamano: {(MART / 'fact_zona.csv').stat().st_size} bytes  (si, bytes).")

---
# ACTO 3 · El mismo dato, dos graficos: uno miente

Ya tenemos el mart. Ahora hay que **enseñárselo** al alcalde. Y aquí es donde se puede mentir
sin querer. Vamos a pintar las alertas por zona de **dos formas**, con **los mismos datos**.

In [ ]:
import matplotlib.pyplot as plt

m = mart.sort_values("n_alertas", ascending=False)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# IZQUIERDA: eje truncado. Empieza en 1500.
ax1.bar(m["nombre"], m["n_alertas"], color="#cf222e")
ax1.set_ylim(1500, 4600)          # <-- el engano
ax1.set_title("Alertas por zona")
ax1.set_ylabel("n alertas")

# DERECHA: honesto. Empieza en 0.
ax2.bar(m["nombre"], m["n_alertas"], color="#1a7f37")
ax2.set_ylim(0, 4600)             # <-- desde 0, siempre
ax2.set_title("Alertas por zona (eje desde 0)")

plt.tight_layout(); plt.show()

Mira los dos. **Son los mismos números.** El de la izquierda hace que el Litoral parezca
**abrumar** al resto — como si el Centro fuera insignificante. El de la derecha cuenta la
verdad: el Litoral y el Centro estan cerca en bruto, y las cinco zonas alertan.

La diferencia es una línea de código: `ylim(1500, ...)` en vez de `ylim(0, ...)`.

> **El eje truncado es el engaño mas común del mundo** (§5.5), y casi nunca se hace con mala fe:
> se hace por descuido, y engaña igual. **En barras, el eje empieza en 0. Siempre.** (En lineas
> de tiempo, no; ahí es otra historia.) La prueba del algodón: enseñáselo a alguien que no
> conozca los datos. Si entiende algo falso, tu gráfico miente — da igual tu intención. **En
> comunicación, cuenta el efecto, no la intención.**

Y el título. `"Alertas por zona"` es un **rótulo**. Un titular seria: *"El Litoral y el Centro
concentran la mitad de las alertas"*. El título enuncia **la conclusión**, no describe el eje.

---
# ACTO 4 · El salto que vale 30 puntos: de la observacion a la decision

Tenemos el mart, tenemos un gráfico honesto. Le enseño al alcalde el gráfico y le digo:

In [ ]:
top = mart.sort_values("n_alertas", ascending=False).iloc[0]
print(f'OBSERVACION: "El {top["nombre"]} tiene {top["n_alertas"]:,} alertas, el maximo de la ciudad."')

**Correcto. Verdadero. E inutil.**

4.387 es mucho? Comparado con qué? El alcalde no puede hacer **nada** con esa frase. Y aquí es
donde el 90 % de los dashboards del mundo se paran, creyendo que hacen BI. Vamos a subir un
escalón. La clave es una sola palabra: **comparar**.

In [ ]:
tot_ev = mart["n_eventos"].sum()
tot_al = mart["n_alertas"].sum()
mart["tasa_alerta_pct"] = (100 * mart["n_alertas"] / mart["n_eventos"]).round(2)
mart["pct_actividad"]   = (100 * mart["n_eventos"] / tot_ev).round(1)
mart["pct_alertas"]     = (100 * mart["n_alertas"] / tot_al).round(1)

print(mart[["nombre", "pct_actividad", "pct_alertas", "tasa_alerta_pct"]].to_string(index=False))

**Ahora el dato habla.** Mira el Litoral frente al Centro:

- El **Centro** tiene el **34 % de la actividad** de la ciudad y genera el 26 % de las alertas.
  Su tasa: **0,61 %**. Mucho trafico, muchas alertas: *normal*.
- El **Litoral** tiene solo el **8 % de la actividad**... y genera casi el **28 % de las
  alertas**. Su tasa: **2,76 %** — **mas de cuatro veces** la del Centro.

En bruto parecían parecidos. **Por tasa, el Litoral es un problema de otra naturaleza.** Tiene
muy poco tráfico y aún así alerta sin parar: eso no es circulación, es algo **estructural** —una
industria, una fuente fija—. El Centro no necesita presupuesto de aire nuevo; necesita gestión
de tráfico, que ya tiene. **El Litoral si.**

> Qué ha cambiado del número bruto al insight? **La comparación.** Un número solo no es
> información; un número **contra una referencia** sí. Y fíjate: el trabajo no fue técnico —fue
> **elegir contra que comparar** (la actividad de cada zona)—. El Centro no estaba en la
> observación; lo traje yo, porque hacia falta un **control**.

Y todavía no hemos terminado. Un insight tampoco decide. Falta el último escalón:

In [ ]:
lit = mart[mart["nombre"] == "Litoral"].iloc[0]
print("DECISION ACCIONABLE (lo que el alcalde puede firmar):\n")
print(f'''  "Destinar el nuevo presupuesto de calidad del aire a una auditoría de
  fuentes fijas en el {lit['nombre']}, cuya tasa de alerta ({lit['tasa_alerta_pct']}%) cuadruplica
  la del resto de la ciudad pese a concentrar solo el {lit['pct_actividad']}% de la actividad.
  Responsable: Medio Ambiente. Objetivo: bajar su tasa al 1% en 12 meses.
  Se medira con este mismo KPI cada trimestre."''')

**Verbo. Responsable. Plazo. Efecto esperado. Cómo se comprueba.** Eso es una decisión.

Los tres niveles, que son 30 de los 100 puntos de tu rúbrica:

| Nivel | Aqui | Quien lo hace |
|---|---|---|
| **Observación** | "El Litoral tiene 4.387 alertas" | El código. Sale solo. |
| **Insight** | "Tasa 2,76 % vs 0,61 %, con 1/4 de la actividad" | **Tu**, eligiendo la comparación. |
| **Decisión** | "Auditar fuentes fijas, Medio Ambiente, 12 meses" | **Tu, con el negocio.** |

> **La prueba del algodón (§5.6):** si tu frase se puede leer sin que nadie tenga que hacer nada
> distinto manana, sigues en el nivel de observación. Y otra: **podrías estar equivocado?** Una
> observación no puede (es un hecho). Una decisión **sí** — y que puedas equivocarte es la señal
> de que has aportado algo. El analista que solo produce observaciones nunca se equivoca, y por
> eso **nunca hace falta**.

---
# ACTO 5 · Y la herramienta, donde queda?

Fíjate en lo que **no** hemos hecho hoy: abrir Power BI ni Tableau. Y sin embargo, el trabajo de
BI esta casi todo hecho. Porque la herramienta es la parte facil:

- Le das el **`fact_zona.csv`** de 5 filas que construimos en el ACTO 2 (no los 2 millones).
- Arrastras `nombre` a un eje y `tasa_alerta_pct` al otro. Barras. **Eje desde 0.**
- Pones de titulo el insight del ACTO 4, no un rótulo.

Eso son diez minutos en cualquiera de las dos herramientas. Lo difícil —el data mart, la
comparación, el salto a la decisión— **ya lo has hecho aquí, en código**.

| | Power BI | Tableau Public |
|---|---|---|
| Precio | De pago (hay capa gratis limitada) | **Gratis** |
| Sistema | **Solo Windows** para crear | **Mac y Windows** |
| Ojo | — | **Tu trabajo se publica en internet.** No subas datos sensibles. |

> En tu actividad eliges **una** de las dos. La demo es neutral a propósito: el `fact_zona.csv`
> vale para ambas. **La herramienta cambia; el data mart y el razonamiento, no.**

---

## Lo que ha pasado en esta hora — y en el módulo

| Acto | La idea | Teoria |
|---|---|---|
| 1 | El data lake crudo mata el dashboard (y el análisis) | §5.3 |
| 2 | El Data Mart: hecho + dimensión + grano. 2M -> 5 filas | §5.3 |
| 3 | El mismo dato, dos gráficos: el eje truncado miente | §5.5 |
| 4 | Observación -> insight -> decisión (la comparación) | §5.6 |
| 5 | La herramienta es lo fácil; el pensamiento es lo tuyo | §5.7 |

Y esto cierra el arco entero: **NF1** lo guardaste, **NF2** lo procesaste, **NF3** lo hiciste
fiable, **NF4** lo vigilaste... y **NF5** lo convertiste en una decisión que alguien firma. La
ingeniería servía para esto. El dato no decide: **tú decides**. Por eso puedes equivocarte, y
por eso haces falta.

> **Cierre del modulo:** en el examen final os pediré exactamente este salto —de un número a una
> decisión defendible—. No es la parte técnica la que separa el aprobado del sobresaliente. Es
> esta.